#### Importing Dependencies:

In [103]:
from bs4 import BeautifulSoup
import requests
import time
import random
import csv
import re

In [104]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'
}

'''I use headers, to make my scraping look more legitimate and reduce risks of IP blocks etc.'''

#### Cleaning Poetry Text:

In [105]:
def clean_poem_text(text):
    if not text or text == 'N/A':
        return 'N/A'

    lines = text.split('\n')
    cleaned_lines = []

    blacklist_phrases = [
        'Add to anthology',
        'Load audio player',
        'This poem is in the public domain.',
        'All rights reserved',
        'Published in the United States',
        'Published in Canada',
        'Copyright',
        'From ',
        '©',
        '×',
    ]

    for line in lines:
        line = line.strip()
        if not line:
            cleaned_lines.append('')
            continue

        # blacklisted phrases
        if any(phrase.lower() in line.lower() for phrase in blacklist_phrases):
            continue

        # lines that look like dates
        if re.match(r'^\d{4}\s?[–-]\s?\d{0,4}$', line):  
            continue

        # lines that are just a year 
        if re.match(r'^\d{4}$', line):
            continue

        cleaned_lines.append(line)

    # rid off multiple blank lines
    cleaned_text = '\n'.join(cleaned_lines)
    cleaned_text = re.sub(r'\n{3,}', '\n\n', cleaned_text)

    return cleaned_text.strip()

In [106]:
poems = []

#### Scraping the Data:

In [107]:
with open('poems.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['title', 'author', 'date', 'text'])
    writer.writeheader()

In [108]:
for page_num in range(1, 2):           
    # page 1 only
    url = f'https://poets.org/poems?page={page_num}'
    soup = BeautifulSoup(requests.get(url, headers=headers).text, 'lxml')

    for row in soup.find_all('tr')[1:]:           
        #skips header row
        title_td = row.find('td', class_='views-field-title')
        if not title_td: continue                 
        a_tag = title_td.find('a')
        if not a_tag: continue

        title  = a_tag.get_text(strip=True)
        link   = 'https://poets.org' + a_tag['href']
        author = (row.find('td', class_='views-field-field-author')
                     .get_text(strip=True))

        time.sleep(random.uniform(2,4))          
        # delays to stay 'ethical'

        page = BeautifulSoup(requests.get(link, headers=headers).text, 'lxml')
        poem_div = page.find('div', id='block-stanza-content')
        if not poem_div: continue
        poem_text = clean_poem_text(poem_div.get_text(separator='\n'))

        poems.append({'title': title, 'author': author, 'text': poem_text})
        print(f"Got: {title} by {author}")

        if len(poems) >= 10:
            with open('poems.csv', 'a', newline='', encoding='utf-8') as f:
                csv.DictWriter(f, fieldnames=fieldnames).writerows(poems)
            poems = []
        # save every 10 poems to array and csv

Got: Mr. Macklin’s Jack O’Lantern by David McCord


KeyboardInterrupt: 

#### Save title, author and poem text to CSV:

In [ ]:
with open('poems.csv', 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['title', 'author', 'text'])
        writer.writeheader()
        writer.writerows(poems)

print("saved to poems.csv")